In [ ]:
import os
import json
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, countDistinct, when, isnan, desc, avg, min, max,
    explode, split, lower, from_unixtime, year, to_date, datediff,
    expr, stddev
)

os.makedirs("/workspace/outputs/figures", exist_ok=True)
os.makedirs("/workspace/outputs/metrics", exist_ok=True)
os.makedirs("/workspace/outputs/tables", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("02_EDA_MovieLens_Spark")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

HDFS_RAW = "hdfs://namenode:8020/netflix-recsys/raw/ml-25m"

df_ratings = spark.read.parquet(f"{HDFS_RAW}/ratings").cache()
df_movies = spark.read.parquet(f"{HDFS_RAW}/movies").cache()
df_tags = spark.read.parquet(f"{HDFS_RAW}/tags").cache()
df_links = spark.read.parquet(f"{HDFS_RAW}/links").cache()

In [ ]:
dataset_overview = {
    "ratings_rows": df_ratings.count(),
    "movies_rows": df_movies.count(),
    "tags_rows": df_tags.count(),
    "links_rows": df_links.count(),
    "ratings_columns": df_ratings.columns,
    "movies_columns": df_movies.columns,
    "tags_columns": df_tags.columns,
    "links_columns": df_links.columns,
}

with open("/workspace/outputs/metrics/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(dataset_overview, f, ensure_ascii=False, indent=2)

pd.DataFrame([
    {"dataset": "ratings", "rows": dataset_overview["ratings_rows"], "columns": len(dataset_overview["ratings_columns"])},
    {"dataset": "movies", "rows": dataset_overview["movies_rows"], "columns": len(dataset_overview["movies_columns"])},
    {"dataset": "tags", "rows": dataset_overview["tags_rows"], "columns": len(dataset_overview["tags_columns"])},
    {"dataset": "links", "rows": dataset_overview["links_rows"], "columns": len(dataset_overview["links_columns"])},
]).to_csv("/workspace/outputs/tables/dataset_overview.csv", index=False)

dataset_overview

In [ ]:
def null_count_expr(df, column_name):
    dtype = dict(df.dtypes)[column_name]
    if dtype in ["double", "float"]:
        return count(
            when(col(column_name).isNull() | isnan(col(column_name)), column_name)
        ).alias(column_name)
    return count(
        when(col(column_name).isNull(), column_name)
    ).alias(column_name)


ratings_nulls = df_ratings.select([null_count_expr(df_ratings, c) for c in df_ratings.columns]).toPandas()
movies_nulls = df_movies.select([null_count_expr(df_movies, c) for c in df_movies.columns]).toPandas()
tags_nulls = df_tags.select([null_count_expr(df_tags, c) for c in df_tags.columns]).toPandas()
links_nulls = df_links.select([null_count_expr(df_links, c) for c in df_links.columns]).toPandas()

ratings_nulls.to_csv("/workspace/outputs/tables/ratings_null_counts.csv", index=False)
movies_nulls.to_csv("/workspace/outputs/tables/movies_null_counts.csv", index=False)
tags_nulls.to_csv("/workspace/outputs/tables/tags_null_counts.csv", index=False)
links_nulls.to_csv("/workspace/outputs/tables/links_null_counts.csv", index=False)

null_summary = {
    "ratings": ratings_nulls.to_dict(orient="records")[0],
    "movies": movies_nulls.to_dict(orient="records")[0],
    "tags": tags_nulls.to_dict(orient="records")[0],
    "links": links_nulls.to_dict(orient="records")[0],
}

with open("/workspace/outputs/metrics/null_summary.json", "w", encoding="utf-8") as f:
    json.dump(null_summary, f, ensure_ascii=False, indent=2)

null_summary

In [ ]:
orphan_ratings = df_ratings.join(df_movies.select("movieId"), on="movieId", how="left_anti")
orphan_tags = df_tags.join(df_movies.select("movieId"), on="movieId", how="left_anti")

invalid_ratings = df_ratings.filter((col("rating") < 0.5) | (col("rating") > 5.0))

duplicate_ratings = (
    df_ratings
    .groupBy("userId", "movieId", "timestamp")
    .agg(count("*").alias("n"))
    .filter(col("n") > 1)
)

data_quality = {
    "orphan_rating_rows": orphan_ratings.count(),
    "orphan_tag_rows": orphan_tags.count(),
    "invalid_rating_rows": invalid_ratings.count(),
    "duplicate_rating_keys": duplicate_ratings.count(),
}

with open("/workspace/outputs/metrics/data_quality_metrics.json", "w", encoding="utf-8") as f:
    json.dump(data_quality, f, ensure_ascii=False, indent=2)

pd.DataFrame([data_quality]).to_csv("/workspace/outputs/tables/data_quality_metrics.csv", index=False)

data_quality

In [ ]:
num_ratings = df_ratings.count()
num_users = df_ratings.select("userId").distinct().count()
num_movies_total = df_movies.select("movieId").distinct().count()
num_movies_rated = df_ratings.select("movieId").distinct().count()

sparsity = 1.0 - (num_ratings / (num_users * num_movies_total))

sparsity_metrics = {
    "num_users": num_users,
    "num_movies_total": num_movies_total,
    "num_movies_rated": num_movies_rated,
    "num_ratings": num_ratings,
    "user_item_matrix_size": num_users * num_movies_total,
    "sparsity": sparsity,
    "sparsity_percent": sparsity * 100,
}

with open("/workspace/outputs/metrics/sparsity_metrics.json", "w", encoding="utf-8") as f:
    json.dump(sparsity_metrics, f, ensure_ascii=False, indent=2)

pd.DataFrame([sparsity_metrics]).to_csv("/workspace/outputs/tables/sparsity_metrics.csv", index=False)

sparsity_metrics

In [ ]:
rating_stats = (
    df_ratings
    .select(
        count("*").alias("count"),
        avg("rating").alias("mean_rating"),
        stddev("rating").alias("std_rating"),
        min("rating").alias("min_rating"),
        expr("percentile_approx(rating, 0.25)").alias("q1_rating"),
        expr("percentile_approx(rating, 0.50)").alias("median_rating"),
        expr("percentile_approx(rating, 0.75)").alias("q3_rating"),
        max("rating").alias("max_rating")
    )
    .toPandas()
)

rating_stats.to_csv("/workspace/outputs/tables/rating_descriptive_statistics.csv", index=False)

rating_stats

In [ ]:
rating_dist_df = (
    df_ratings
    .groupBy("rating")
    .agg(count("*").alias("count"))
    .orderBy("rating")
    .toPandas()
)

rating_dist_df.to_csv("/workspace/outputs/tables/rating_distribution.csv", index=False)

plt.figure(figsize=(10, 5))
plt.bar(rating_dist_df["rating"].astype(str), rating_dist_df["count"])
plt.title("Phân phối các mức điểm đánh giá trong MovieLens 25M", fontsize=14)
plt.xlabel("Mức điểm rating", fontsize=12)
plt.ylabel("Số lượng đánh giá", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/rating_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
user_activity = (
    df_ratings
    .groupBy("userId")
    .agg(count("*").alias("num_ratings"))
    .cache()
)

user_activity_stats = (
    user_activity
    .select(
        count("*").alias("num_users"),
        avg("num_ratings").alias("avg_ratings_per_user"),
        stddev("num_ratings").alias("std_ratings_per_user"),
        min("num_ratings").alias("min_ratings_per_user"),
        expr("percentile_approx(num_ratings, 0.25)").alias("q1_ratings_per_user"),
        expr("percentile_approx(num_ratings, 0.50)").alias("median_ratings_per_user"),
        expr("percentile_approx(num_ratings, 0.75)").alias("q3_ratings_per_user"),
        max("num_ratings").alias("max_ratings_per_user")
    )
    .toPandas()
)

user_activity_stats.to_csv("/workspace/outputs/tables/user_activity_statistics.csv", index=False)

user_activity_plot = (
    user_activity
    .withColumn(
        "rating_group",
        when(col("num_ratings") < 20, "<20")
        .when(col("num_ratings") < 50, "20-49")
        .when(col("num_ratings") < 100, "50-99")
        .when(col("num_ratings") < 200, "100-199")
        .when(col("num_ratings") < 500, "200-499")
        .otherwise("500+")
    )
    .groupBy("rating_group")
    .agg(count("*").alias("count"))
    .toPandas()
)

order = ["<20", "20-49", "50-99", "100-199", "200-499", "500+"]
user_activity_plot["rating_group"] = pd.Categorical(user_activity_plot["rating_group"], categories=order, ordered=True)
user_activity_plot = user_activity_plot.sort_values("rating_group")

user_activity_plot.to_csv("/workspace/outputs/tables/user_activity_distribution.csv", index=False)

plt.figure(figsize=(10, 5))
plt.bar(user_activity_plot["rating_group"].astype(str), user_activity_plot["count"])
plt.title("Phân phối số lượng rating trên mỗi người dùng", fontsize=14)
plt.xlabel("Nhóm số lượng rating/user", fontsize=12)
plt.ylabel("Số lượng user", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/ratings_per_user_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

user_activity_stats

In [ ]:
movie_activity = (
    df_ratings
    .groupBy("movieId")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
    .cache()
)

movie_activity_stats = (
    movie_activity
    .select(
        count("*").alias("num_movies_rated"),
        avg("num_ratings").alias("avg_ratings_per_movie"),
        stddev("num_ratings").alias("std_ratings_per_movie"),
        min("num_ratings").alias("min_ratings_per_movie"),
        expr("percentile_approx(num_ratings, 0.25)").alias("q1_ratings_per_movie"),
        expr("percentile_approx(num_ratings, 0.50)").alias("median_ratings_per_movie"),
        expr("percentile_approx(num_ratings, 0.75)").alias("q3_ratings_per_movie"),
        max("num_ratings").alias("max_ratings_per_movie")
    )
    .toPandas()
)

movie_activity_stats.to_csv("/workspace/outputs/tables/movie_activity_statistics.csv", index=False)

movie_activity_plot = (
    movie_activity
    .withColumn(
        "rating_group",
        when(col("num_ratings") < 10, "<10")
        .when(col("num_ratings") < 50, "10-49")
        .when(col("num_ratings") < 100, "50-99")
        .when(col("num_ratings") < 500, "100-499")
        .when(col("num_ratings") < 1000, "500-999")
        .otherwise("1000+")
    )
    .groupBy("rating_group")
    .agg(count("*").alias("count"))
    .toPandas()
)

order = ["<10", "10-49", "50-99", "100-499", "500-999", "1000+"]
movie_activity_plot["rating_group"] = pd.Categorical(movie_activity_plot["rating_group"], categories=order, ordered=True)
movie_activity_plot = movie_activity_plot.sort_values("rating_group")

movie_activity_plot.to_csv("/workspace/outputs/tables/movie_activity_distribution.csv", index=False)

plt.figure(figsize=(10, 5))
plt.bar(movie_activity_plot["rating_group"].astype(str), movie_activity_plot["count"])
plt.title("Phân phối số lượng rating trên mỗi phim", fontsize=14)
plt.xlabel("Nhóm số lượng rating/movie", fontsize=12)
plt.ylabel("Số lượng phim", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/ratings_per_movie_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

movie_activity_stats

In [ ]:
top_movies_by_count = (
    movie_activity
    .join(df_movies, "movieId", "left")
    .orderBy(desc("num_ratings"))
    .limit(20)
    .toPandas()
)

top_movies_by_count.to_csv("/workspace/outputs/tables/top_20_movies_by_rating_count.csv", index=False)

plot_df = top_movies_by_count.sort_values("num_ratings")

plt.figure(figsize=(10, 8))
plt.barh(plot_df["title"], plot_df["num_ratings"])
plt.title("Top 20 phim có nhiều lượt đánh giá nhất", fontsize=14)
plt.xlabel("Số lượng rating", fontsize=12)
plt.ylabel("Tên phim", fontsize=12)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/top_20_movies_by_rating_count.png", dpi=200, bbox_inches="tight")
plt.show()

top_movies_by_count

In [ ]:
top_movies_by_avg_rating = (
    movie_activity
    .filter(col("num_ratings") >= 1000)
    .join(df_movies, "movieId", "left")
    .orderBy(desc("avg_rating"))
    .limit(20)
    .toPandas()
)

top_movies_by_avg_rating.to_csv("/workspace/outputs/tables/top_20_movies_by_avg_rating.csv", index=False)

plot_df = top_movies_by_avg_rating.sort_values("avg_rating")

plt.figure(figsize=(10, 8))
plt.barh(plot_df["title"], plot_df["avg_rating"])
plt.title("Top 20 phim có điểm trung bình cao nhất với tối thiểu 1000 rating", fontsize=14)
plt.xlabel("Điểm rating trung bình", fontsize=12)
plt.ylabel("Tên phim", fontsize=12)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/top_20_movies_by_avg_rating.png", dpi=200, bbox_inches="tight")
plt.show()

top_movies_by_avg_rating

In [ ]:
genres_df = (
    df_movies
    .withColumn("genre", explode(split(col("genres"), "\\|")))
    .filter(col("genre") != "(no genres listed)")
)

genre_distribution = (
    genres_df
    .groupBy("genre")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
    .toPandas()
)

genre_distribution.to_csv("/workspace/outputs/tables/genre_distribution.csv", index=False)

plot_df = genre_distribution.sort_values("count")

plt.figure(figsize=(10, 7))
plt.barh(plot_df["genre"], plot_df["count"])
plt.title("Số lượng phim theo từng thể loại", fontsize=14)
plt.xlabel("Số lượng phim", fontsize=12)
plt.ylabel("Thể loại", fontsize=12)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/genre_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

genre_distribution

In [ ]:
genre_rating = (
    df_ratings
    .join(genres_df.select("movieId", "genre"), "movieId", "inner")
    .groupBy("genre")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
    .orderBy(desc("avg_rating"))
    .toPandas()
)

genre_rating.to_csv("/workspace/outputs/tables/genre_rating_statistics.csv", index=False)

plot_df = genre_rating.sort_values("avg_rating")

plt.figure(figsize=(10, 7))
plt.barh(plot_df["genre"], plot_df["avg_rating"])
plt.title("Điểm rating trung bình theo thể loại phim", fontsize=14)
plt.xlabel("Rating trung bình", fontsize=12)
plt.ylabel("Thể loại", fontsize=12)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/avg_rating_by_genre.png", dpi=200, bbox_inches="tight")
plt.show()

genre_rating

In [ ]:
top_tags_df = (
    df_tags
    .filter(col("tag").isNotNull())
    .withColumn("tag_lower", lower(col("tag")))
    .groupBy("tag_lower")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
    .limit(15)
    .toPandas()
)

top_tags_df.to_csv("/workspace/outputs/tables/top_15_user_tags.csv", index=False)

plot_df = top_tags_df.sort_values("count")

plt.figure(figsize=(10, 6))
plt.barh(plot_df["tag_lower"], plot_df["count"])
plt.title("Top 15 tags được người dùng sử dụng nhiều nhất", fontsize=14)
plt.xlabel("Số lượt gắn thẻ", fontsize=12)
plt.ylabel("Tag", fontsize=12)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/top_15_user_tags.png", dpi=200, bbox_inches="tight")
plt.show()

top_tags_df

In [ ]:
ratings_by_year = (
    df_ratings
    .withColumn("rating_datetime", from_unixtime(col("timestamp")))
    .withColumn("rating_year", year(col("rating_datetime")))
    .groupBy("rating_year")
    .agg(count("*").alias("num_ratings"))
    .orderBy("rating_year")
    .toPandas()
)

ratings_by_year.to_csv("/workspace/outputs/tables/ratings_by_year.csv", index=False)

plt.figure(figsize=(11, 5))
plt.plot(ratings_by_year["rating_year"], ratings_by_year["num_ratings"], marker="o")
plt.title("Số lượng rating theo năm", fontsize=14)
plt.xlabel("Năm", fontsize=12)
plt.ylabel("Số lượng rating", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/ratings_by_year.png", dpi=200, bbox_inches="tight")
plt.show()

ratings_by_year

In [ ]:
user_lifetime = (
    df_ratings
    .withColumn("rating_date", to_date(from_unixtime(col("timestamp"))))
    .groupBy("userId")
    .agg(
        count("*").alias("num_ratings"),
        min("rating_date").alias("first_rating_date"),
        max("rating_date").alias("last_rating_date"),
        avg("rating").alias("avg_user_rating")
    )
    .withColumn("active_days", datediff(col("last_rating_date"), col("first_rating_date")))
    .cache()
)

long_term_users = (
    user_lifetime
    .filter((col("num_ratings") >= 50) & (col("active_days") >= 180))
)

long_term_summary = {
    "total_users": user_lifetime.count(),
    "long_term_users": long_term_users.count(),
    "long_term_user_ratio": long_term_users.count() / user_lifetime.count(),
}

long_term_stats = (
    long_term_users
    .select(
        count("*").alias("num_long_term_users"),
        avg("num_ratings").alias("avg_ratings_per_long_term_user"),
        min("num_ratings").alias("min_ratings_per_long_term_user"),
        expr("percentile_approx(num_ratings, 0.5)").alias("median_ratings_per_long_term_user"),
        max("num_ratings").alias("max_ratings_per_long_term_user"),
        avg("active_days").alias("avg_active_days"),
        min("active_days").alias("min_active_days"),
        max("active_days").alias("max_active_days")
    )
    .toPandas()
)

with open("/workspace/outputs/metrics/long_term_user_summary.json", "w", encoding="utf-8") as f:
    json.dump(long_term_summary, f, ensure_ascii=False, indent=2)

long_term_stats.to_csv("/workspace/outputs/tables/long_term_user_statistics.csv", index=False)

long_term_plot = (
    user_lifetime
    .withColumn(
        "user_group",
        when((col("num_ratings") >= 50) & (col("active_days") >= 180), "Long-term users")
        .otherwise("Other users")
    )
    .groupBy("user_group")
    .agg(count("*").alias("count"))
    .toPandas()
)

long_term_plot.to_csv("/workspace/outputs/tables/long_term_user_distribution.csv", index=False)

plt.figure(figsize=(7, 5))
plt.bar(long_term_plot["user_group"], long_term_plot["count"])
plt.title("Phân bố người dùng dài hạn và người dùng còn lại", fontsize=14)
plt.xlabel("Nhóm người dùng", fontsize=12)
plt.ylabel("Số lượng user", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("/workspace/outputs/figures/long_term_user_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

long_term_summary, long_term_stats

In [ ]:
eda_final_summary = {
    "dataset_overview": dataset_overview,
    "data_quality": data_quality,
    "sparsity_metrics": sparsity_metrics,
    "long_term_user_summary": long_term_summary,
}

with open("/workspace/outputs/metrics/eda_final_summary.json", "w", encoding="utf-8") as f:
    json.dump(eda_final_summary, f, ensure_ascii=False, indent=2)

eda_final_summary